# Исследование датасетов синтезированной речи

Notebook получает только небольшие официальные protocol/metadata-файлы и **не скачивает аудиокорпуса автоматически**. Большие загрузки защищены явным флагом подтверждения. Актуальность проверена 21.11.2026; ссылки и ограничения зафиксированы в `docs/DATASET_COMPARISON.md`.

In [ ]:
# 1. Проверка среды
import platform
import sys
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Рабочий каталог:', Path.cwd())

## Подготовка Colab
В Colab сначала клонируйте репозиторий и выполните `%cd audio-deepfake-detector`. Затем установите исследовательские зависимости: `!pip install -r requirements-research.txt`.

In [ ]:
# 2. Импорты
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT.parent))

from audio_deepfake_detector.research.dataset_metadata import (
    DATASET_DOWNLOADS,
    analyse_asvspoof5,
    download_file,
    extract_tar_safely,
    fetch_huggingface_tree,
    fetch_ruasd_preview,
    format_bytes,
    load_asvspoof5_protocol,
    load_asvspoof2021_df_metadata,
)

sns.set_theme(style='whitegrid')
CACHE = PROJECT_ROOT / '.dataset_research_cache'
CACHE.mkdir(exist_ok=True)

In [ ]:
# 3. План загрузок: сначала смотрим размер, затем решаем
download_plan = pd.DataFrame([
    {
        'id': key, 'name': spec.name,
        'estimated_size': format_bytes(spec.expected_bytes),
        'metadata_only': spec.metadata_only, 'url': spec.url,
    }
    for key, spec in DATASET_DOWNLOADS.items()
])
display(download_plan)

## ASVspoof 5: официальный protocol отдельно от аудио
Архив protocol имеет размер около 20,7 MB. Официальные поля Track 1: `SPEAKER_ID FLAC_FILE_NAME SPEAKER_GENDER CODEC CODEC_Q CODEC_SEED ATTACK_TAG ATTACK_LABEL KEY TMP`. В README источника они по ошибке названы «FIVE columns», хотя перечислено и фактически присутствует 10 полей.

In [ ]:
# 4. Загрузка только официального ASVspoof 5 protocol (не аудио)
spec = DATASET_DOWNLOADS['asvspoof5_protocols']
archive = CACHE / 'ASVspoof5_protocols.tar.gz'
if not archive.exists():
    download_file(spec.url, archive, expected_bytes=spec.expected_bytes)
protocol_dir = CACHE / 'asvspoof5_protocols'
extract_tar_safely(archive, protocol_dir)
sorted(path.name for path in protocol_dir.iterdir())

In [ ]:
# 5. Точные распределения Track 1
asv5 = {
    'train': load_asvspoof5_protocol(protocol_dir / 'ASVspoof5.train.tsv'),
    'dev': load_asvspoof5_protocol(protocol_dir / 'ASVspoof5.dev.track_1.tsv'),
    'eval': load_asvspoof5_protocol(protocol_dir / 'ASVspoof5.eval.track_1.tsv'),
}
summary = analyse_asvspoof5(asv5)
display(summary['splits'])
display(summary['attacks'])
display(summary['codecs'])

In [ ]:
# 6. Дополнительные реальные поля protocol
for split, frame in asv5.items():
    print(f'\n{split}: {len(frame):,} файлов; {frame.speaker_id.nunique():,} speaker_id')
    display(frame.head())
    display(frame['gender'].value_counts().rename('count').to_frame())
    display(frame['attack_tag'].value_counts().rename('count').to_frame())

In [ ]:
# 7. Графики REAL/FAKE, атак и кодеков
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
label_long = pd.concat([f.assign(split=s) for s, f in asv5.items()])
sns.countplot(data=label_long, x='split', hue='key', ax=axes[0])
sns.barplot(data=summary['attacks'], x='attack_label', y='files', hue='split', ax=axes[1])
axes[1].tick_params(axis='x', rotation=90)
codec_eval = summary['codecs'].query("split == 'eval' and codec != '-'")
sns.barplot(data=codec_eval, x='codec', y='files', ax=axes[2])
axes[2].tick_params(axis='x', rotation=45)
axes[0].set_title('ASVspoof 5: REAL/FAKE')
axes[1].set_title('Attack ID по разделам')
axes[2].set_title('Кодеки eval')
plt.tight_layout()

## ASVspoof 2021 DF
Официальный архив `DF-keys-full.tar.gz` (≈26 MB) доступен отдельно от 34,5 GB аудио. Его можно загрузить следующей ячейкой; полный аудионабор не загружается. На сервере ASVspoof скорость иногда низкая.

In [ ]:
# 8. Необязательная загрузка только keys/metadata ASVspoof 2021 DF
DOWNLOAD_ASVSPOOF2021_KEYS = False
if DOWNLOAD_ASVSPOOF2021_KEYS:
    spec = DATASET_DOWNLOADS['asvspoof2021_df_keys']
    target = CACHE / 'DF-keys-full.tar.gz'
    download_file(spec.url, target, expected_bytes=spec.expected_bytes, timeout=600)
    keys_dir = CACHE / 'asvspoof2021_df_keys'
    extract_tar_safely(target, keys_dir)
    df21 = load_asvspoof2021_df_metadata(keys_dir / 'keys/DF/CM/trial_metadata.txt')
    print('Сохранено:', target, format_bytes(target.stat().st_size))
    print('Строк:', f'{len(df21):,}', 'Дикторов:', df21.speaker.nunique())
    for column in ['label', 'subset', 'source', 'codec', 'vocoder', 'attack', 'language']:
        display(df21[column].value_counts().rename('count').to_frame())
else:
    print('Пропущено. Установите DOWNLOAD_ASVSPOOF2021_KEYS=True для metadata без аудио.')

## Русский язык: MLAAD и RuASD
MLAAD v11 содержит русский и множество современных TTS/VC, но HF-репозиторий сейчас gated: список файлов виден без токена, чтение `meta.csv` требует принятия лицензии и `HF_TOKEN`. RuASD позволяет просмотреть небольшое число metadata-строк через Dataset Viewer без загрузки ≈250 GB аудио.

In [ ]:
# 9. Какие русские генераторы реально видны в текущем MLAAD (без аудио)
mlaad_ru_tree = fetch_huggingface_tree('mueller91/MLAAD', subpath='fake/ru')
ru_models = sorted(
    item['path'].split('/')[-1]
    for item in mlaad_ru_tree
    if item.get('type') == 'directory'
)
print('Русских model directories:', len(ru_models))
display(pd.DataFrame({'model_directory': ru_models}))

In [ ]:
# 10. Небольшая metadata-выборка RuASD; аудио не скачивается
ruasd_preview = fetch_ruasd_preview(length=20)
display(ruasd_preview.head())
for column in ['label', 'group', 'subset', 'source_type', 'model']:
    if column in ruasd_preview:
        display(ruasd_preview[column].value_counts(dropna=False).rename('count').to_frame())
print('Важно: preview не является статистически репрезентативной полной выборкой.')

## Явное подтверждение большой загрузки
Следующая ячейка по умолчанию ничего не скачивает. Даже при включении флага функция потребует `confirm_large=True`. Для ASVspoof 5 лучше скачивать отдельные официальные tar-части и заранее выбрать файлы по protocol; Zenodo не поддерживает выбор отдельных FLAC внутри tar.

In [ ]:
# 11. Шаблон, а не автоматическая загрузка
CONFIRM_LARGE_DOWNLOAD = False
selected = DATASET_DOWNLOADS['in_the_wild']
print(selected.name, format_bytes(selected.expected_bytes), selected.url)
if CONFIRM_LARGE_DOWNLOAD:
    download_file(
        selected.url, CACHE / 'release_in_the_wild.zip',
        expected_bytes=selected.expected_bytes,
        confirm_large=True, timeout=3600,
    )
else:
    print('Загрузка НЕ выполнена. Проверьте свободное место и лицензию.')

In [ ]:
# 12. Таблица решения для первого обучения CNN
decision = pd.DataFrame([
    ['ASVspoof 2019 LA train', 'TRAIN baseline', 'малый, стабильный protocol', 'старые атаки; English'],
    ['ASVspoof 2019 LA dev', 'VALIDATION baseline', 'официальный split', 'matched-domain'],
    ['MLAAD v11 Russian subset + M-AILABS real', 'TRAIN extension', 'русский, современные генераторы', 'HF login; real отдельно; leakage audit'],
    ['ASVspoof 2021 DF eval', 'TEST', '100+ атак и кодеки', 'только evaluation'],
    ['In-the-Wild', 'EXTERNAL TEST', 'реальные интернет-условия', 'English, генераторы не размечены'],
    ['RuASD held-out generators', 'RUSSIAN TEST', 'русский и современные TTS', 'нет официального train/dev/test; split проектировать самим'],
    ['Собственные 5 дикторов/RVC', 'FINAL EXTERNAL TEST', 'нужный прикладной сценарий', 'никогда не использовать для train/tuning'],
], columns=['dataset/split', 'role', 'strength', 'limitation'])
display(decision)